In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf

In [ ]:
from tensorflow import keras

In [ ]:
from keras import regularizers
def identity_block(inputs,filters,s=1):
    
    x1=keras.layers.Conv2D(
    filters, 3,    strides=(s, s),padding="same",
    activation=None,use_bias=True,kernel_initializer="glorot_uniform",
    bias_initializer="zeros",kernel_regularizer=regularizers.L2(0.01),)(inputs)
    b1= keras.layers.BatchNormalization(
    axis=-1,    momentum=0.99,epsilon=0.001,    center=True,
    scale=True,beta_initializer="zeros", gamma_initializer="ones",moving_mean_initializer="zeros",
    moving_variance_initializer="ones",)(x1)
    relu1=keras.layers.ReLU(max_value=None, negative_slope=0.01, threshold=0.0)(b1)
    x2=keras.layers.Conv2D(
    filters, 3,    strides=(1, 1),padding="same",
    activation=None,use_bias=True,kernel_initializer="glorot_uniform",
    bias_initializer="zeros",kernel_regularizer=regularizers.L2(0.01),)(relu1)
    b2= keras.layers.BatchNormalization(
    axis=-1,    momentum=0.99,epsilon=0.001,    center=True,
    scale=True,beta_initializer="zeros", gamma_initializer="ones",moving_mean_initializer="zeros",
    moving_variance_initializer="ones",)(x2)
    if s==2:
        inputs=keras.layers.Conv2D(    filters, 3,strides=(s, s),padding="same",activation=None,use_bias=True,     
                                   kernel_initializer="glorot_uniform",bias_initializer="zeros")(inputs)
        

    output= Add()([inputs, b2])
    output=keras.layers.ReLU(max_value=None, negative_slope=0.01, threshold=0.0)(output)
    
    
    return output
    
    



In [ ]:
from tensorflow.keras.layers import Dropout,Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization,Flatten, Conv2D, AveragePooling2D, MaxPooling2D, GlobalAveragePooling2D

from tensorflow.keras.models import Model
def ResNet34(input_shape=(224, 224, 3),num_classes=1000) :
    inputs = tf.keras.Input(shape=input_shape)

    ### Level 1 ###

    # padding
    X = ZeroPadding2D((3, 3))(inputs)

    # convolutional layer, followed by batch normalization and relu activation
    X = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2),            
               kernel_initializer="glorot_uniform")(X)
    X = BatchNormalization(axis=3)(X)
    X = Activation('relu')(X)
    X = ZeroPadding2D((1, 1))(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)
    X=identity_block(X,64)
    X=identity_block(X,64)
    X=identity_block(X,64)
    X = Dropout(0.7)(X)
    
    X=identity_block(X,128,s=2)
    X=identity_block(X,128)
    X=identity_block(X,128)
    X = Dropout(0.5)(X)
    
    X=identity_block(X,256,s=2)
    X=identity_block(X,256)
    X=identity_block(X,256)

    X = Dropout(0.6)(X)
    
    X=identity_block(X,256)
    X=identity_block(X,256)
    X=identity_block(X,256)
    
    X = Dropout(0.7)(X)
    
    
    X=identity_block(X,512,s=2)
    X=identity_block(X,512)
    X = Dropout(0.7)(X)
    X=identity_block(X,512)

    
    
    X= GlobalAveragePooling2D()(X)
    outputs = Dense(256, activation='relu')(X)
    outputs = Dense(num_classes, activation='softmax')(outputs)

    model = Model(inputs, outputs)
    return model



In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential

from IPython.display import display, Image
from keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Define the path to the dataset folders
happy_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/train/happy"
sad_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/train/Sad"
angry_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/train/Angry"
other_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/train/Other"


# Function to load and preprocess images
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        # print(img.shape)
        if img is not None:
            # img = cv2.cvtColor(img)
            img = cv2.resize(img, (224, 224))  # Resize to a fixed size for the model
            images.append(img)
    return images

# Load images and labels for each emotion
happy_images = load_images_from_folder(happy_folder)
sad_images = load_images_from_folder(sad_folder)
angry_images = load_images_from_folder(angry_folder)
other_images = load_images_from_folder(other_folder)


# Create labels for each emotion category
happy_labels = [0] * len(happy_images)
sad_labels = [1] * len(sad_images)
angry_labels = [2] * len(angry_images)
other_labels = [3] * len(other_images)


# Concatenate images and labels
X = np.array(happy_images + sad_images + angry_images+other_images)
y = np.array(happy_labels + sad_labels + angry_labels+other_labels)

# Normalize pixel values to range [0, 1]
X = X.astype('float32') / 255.0

# One-hot encode the labels
y = to_categorical(y, 4)

# Split the data into training and testing sets
X_train, y_train=X,y
print(len(X_train))
train_datagen = ImageDataGenerator(
    rotation_range=20,         # Randomly rotate images by 20 degrees
    width_shift_range=0.2,     # Shift images horizontally by 20% of width
    height_shift_range=0.2,    # Shift images vertically by 20% of height
    shear_range=0.2,           # Apply shearing transformation
    zoom_range=0.2,            # Randomly zoom in/out
    horizontal_flip=True,      # Randomly flip images horizontally
    fill_mode='nearest'        # Fill pixels using nearest value
)

# Fit generator to training data
train_generator = train_datagen.flow(X_train, y_train, batch_size=4)


In [ ]:
# Define the path to the dataset folders
happy_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/valid/happy"
sad_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/valid/Sad"
angry_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/valid/Angry"
other_folder = "/kaggle/input/pets-facial-expression-dataset/Master Folder/valid/Other"


# Function to load and preprocess images
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        # print(img.shape)
        if img is not None:
            # img = cv2.cvtColor(img)
            img = cv2.resize(img, (224, 224))  # Resize to a fixed size for the model
            images.append(img)
    return images

# Load images and labels for each emotion
happy_images = load_images_from_folder(happy_folder)
sad_images = load_images_from_folder(sad_folder)
angry_images = load_images_from_folder(angry_folder)
other_images = load_images_from_folder(other_folder)


# Create labels for each emotion category
happy_labels = [0] * len(happy_images)
sad_labels = [1] * len(sad_images)
angry_labels = [2] * len(angry_images)
other_labels = [3] * len(other_images)


# Concatenate images and labels
X = np.array(happy_images + sad_images + angry_images+other_images)
y = np.array(happy_labels + sad_labels + angry_labels+other_labels)

# Normalize pixel values to range [0, 1]
X = X.astype('float32') / 255.0

# One-hot encode the labels
y = to_categorical(y, 4)


# Split the data into training and testing sets
X_valid, y_valid=X,y

In [ ]:
X_train[0].shape

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW


model = ResNet34(input_shape=(224, 224, 3), num_classes=4)
model.summary()


# Define the AdamW optimizer with weight decay
optimizer = AdamW(learning_rate=0.00009, weight_decay=0.00003)

model.compile(
    optimizer=optimizer, # optimizer
    loss='categorical_crossentropy', # loss function to optimize 
    metrics=['accuracy','precision','recall'] # metrics to monitor
)

In [ ]:
device_name = tf.test.gpu_device_name()
if "GPU" not in device_name:
    print("GPU device not found")
print('Found GPU at: {}'.format(device_name))

In [ ]:
print(X_train.shape, y_train.shape)


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint

# Define the checkpoint callback to save the model every 5 epochs
checkpoint_callback = ModelCheckpoint(
    'model_epoch_{epoch:02d}.keras',  # Saves with the epoch number
    save_freq=10 * len(X_train) // 16,  # Save every 5 epochs (adjust batch size as necessary)
    save_best_only=False,  # Optionally save only the best model
    verbose=1
)

# Fit the model with the callback
with tf.device('/gpu:0'):
    history = model.fit(
        train_generator,
        validation_data=(X_valid, y_valid),
        epochs=50,
        batch_size=32,
        verbose=1,
        callbacks=[checkpoint_callback]
    )

# Plot loss and accuracy over epochs
plt.figure(figsize=(12, 5))

# Plot training & validation loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot training & validation accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Show plots
plt.tight_layout()
plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
resnet=Sequential()
pre_trained=tf.keras.applications.ResNet50(include_top=False,input_shape=(224,224,3),classes=3)
resnet.add(pre_trained)
resnet.add(Flatten())
resnet.add(Dense(256,activation="relu"))
resnet.add(Dense(3,activation="softmax"))

# resnet.summary()
# Define the AdamW optimizer with weight decay
optimizer = AdamW(learning_rate=0.0000001, weight_decay=0.00000000003)

resnet.compile(
    optimizer=optimizer, # optimizer
    loss='categorical_crossentropy', # loss function to optimize 
    metrics=['accuracy','precision','recall'] # metrics to monitor
)
resnet

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint

# Define the checkpoint callback to save the model every 5 epochs
checkpoint_callback = ModelCheckpoint(
    'res_epoch_{epoch:02d}.keras',  # Saves with the epoch number
    save_freq=10 * len(X_train) // 16,  # Save every 5 epochs (adjust batch size as necessary)
    save_best_only=False,  # Optionally save only the best model
    verbose=1
)

# Fit the model with the callback
with tf.device('/gpu:0'):
    history = resnet.fit(
        train_generator,
        validation_data=(X_valid, y_valid),
        epochs=10,
        batch_size=32,
        verbose=1,
        callbacks=[checkpoint_callback]
    )

# Plot loss and accuracy over epochs
plt.figure(figsize=(12, 5))

# Plot training & validation loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot training & validation accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Show plots
plt.tight_layout()
plt.show()
